# 07_train_with_decoys — decoy 균형 데이터로 재학습

**한 줄 요약:** decoy로 1:1 균형 맞춘 학습셋으로 4종 fingerprint 모델을 다시 학습·비교하고, **실측 inactive를 얼마나 맞히는지**(진짜 어려운 지표)도 따로 본 뒤, 최고 모델을 파일로 저장한다.
**큰 흐름:** ① 준비 → ② 데이터·라벨 → ③ 지문 계산 → ④ 5-fold 비교(+실측inactive) → ⑤ 최고 모델 저장

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 불러오기
학습·평가·저장에 필요한 라이브러리를 가져온다.

In [ ]:
import sys
import numpy as np
import pandas as pd
import pickle
from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix)
from lightgbm import LGBMClassifier

🔎 **코드 뜯어보기 (셀 1)**
- `import sys` : 실행할 때 준 **명령행 인자**를 읽는 도구. `import pickle`=파이썬 객체(모델)를 파일로 저장.

### 셀 2 — 학습셋 경로 받기 + 데이터 로드
실행 인자로 학습셋/모델 경로를 받고(기본은 06 결과), 데이터를 읽어 라벨과 출처를 정리한다.

In [ ]:
SRC = sys.argv[1] if len(sys.argv) > 1 else "data/HSD17B13_train_with_decoys.xlsx"
MODEL_OUT = sys.argv[2] if len(sys.argv) > 2 else "data/HSD17B13_screen_model.pkl"
NBITS = 1024
print(f"학습셋: {SRC}")

df = pd.read_excel(SRC).dropna(subset=["canonical_smiles"]).reset_index(drop=True)
print("학습셋 구성:")
print(df.groupby(["label", "source"]).size().to_string())

🔎 **코드 뜯어보기 (셀 2)**
- `sys.argv[1] if len(sys.argv) > 1 else "기본경로"` : **sys.argv**=실행 시 넘긴 값 목록. 값이 있으면 그걸, 없으면 기본 경로를 씀(노트북에선 보통 기본값). `A if 조건 else B`=조건부 값.

### 셀 3 — fingerprint 4종 계산
각 물질을 4가지 지문으로 변환하고, '실측 inactive'가 어디인지 표시(mask)해 둔다.

In [ ]:
gens = {
    "ECFP4": rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS),
    "RDKit": rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NBITS),
    "AtomPair": rdFingerprintGenerator.GetAtomPairGenerator(fpSize=NBITS),
}


def maccs_np(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def featurize(name, mol):
    if name == "MACCS":
        return maccs_np(mol)
    return gens[name].GetFingerprintAsNumPy(mol)


FP_NAMES = ["ECFP4", "MACCS", "RDKit", "AtomPair"]
fps = {n: [] for n in FP_NAMES}
y, src = [], []
for smi, lab, s in zip(df["canonical_smiles"], df["label"], df["source"]):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        continue
    for n in FP_NAMES:
        fps[n].append(featurize(n, mol))
    y.append(int(lab))
    src.append(s)
y = np.array(y)
src = np.array(src)
real_inact = (y == 0) & (src == "real")   # 실측 inactive 마스크
print(f"\n학습 물질 {len(y)}개 | active {int(y.sum())} "
      f"inactive {int((y==0).sum())} (실측 {int(real_inact.sum())} + decoy {int(((y==0)&(src=='decoy')).sum())})")

🔎 **코드 뜯어보기 (셀 3)** *(생성기·featurize는 04·05에서 설명)*
- `real_inact = (y == 0) & (src == "real")` : 정답이 0(inactive)이고 출처가 실측인 것 → **실측 inactive mask**(True/False 배열). 뒤에서 이 부분의 정답률을 따로 잰다.

### 셀 4 — 4종 비교 (5-fold CV) + 실측 inactive 정답률
지문마다 교차검증으로 채점하되, decoy로 부풀지 않는 **실측 inactive 판별력**도 함께 계산한다.

In [ ]:
print("\n" + "=" * 74)
print("Fingerprint별 5-fold CV (LightGBM, class_weight=balanced)")
print("=" * 74)
print(f"{'FP':9s} {'ROC-AUC':>8s} {'PR-AUC':>8s} {'Recall(act)':>12s} "
      f"{'Prec(act)':>10s} {'실측inact 정답률':>16s}")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name in FP_NAMES:
    X = np.vstack(fps[name])
    clf = LGBMClassifier(n_estimators=400, class_weight="balanced",
                         random_state=42, n_jobs=-1, verbosity=-1)
    proba = cross_val_predict(clf, X, y, cv=skf, method="predict_proba",
                              n_jobs=-1)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba)
    pr = average_precision_score(y, proba)
    tn, fp_, fn, tp = confusion_matrix(y, pred).ravel()
    recall = tp / (tp + fn) if (tp + fn) else 0
    prec = tp / (tp + fp_) if (tp + fp_) else 0
    # 실측 inactive를 실제로 inactive(0)로 맞힌 비율
    ri_acc = np.mean(pred[real_inact] == 0) if real_inact.any() else float("nan")
    results[name] = auc
    print(f"{name:9s} {auc:8.3f} {pr:8.3f} {recall:12.3f} {prec:10.3f} {ri_acc:16.3f}")

best = max(results, key=results.get)
print(f"\n>>> 최고 fingerprint: {best} (ROC-AUC {results[best]:.3f})")

🔎 **코드 뜯어보기 (셀 4)** *(cross_val_predict·confusion_matrix는 05에서 설명)*
- `np.mean(pred[real_inact] == 0)` : 실측 inactive 위치의 예측이 0(맞음)인 **비율**. `pred[mask]`=mask가 True인 것만 골라보기.

### 셀 5 — 최고 지문으로 전체 재학습 → 모델 저장
가장 좋은 지문으로 전체 데이터를 학습해 스크리닝용 모델(.pkl)로 저장한다.

In [ ]:
Xbest = np.vstack(fps[best])
final = LGBMClassifier(n_estimators=400, class_weight="balanced",
                       random_state=42, n_jobs=-1, verbosity=-1)
final.fit(Xbest, y)
with open(MODEL_OUT, "wb") as f:
    pickle.dump({"model": final, "fp_name": best, "nbits": NBITS,
                 "n_active": int(y.sum()), "n_inactive": int((y == 0).sum())}, f)
print(f"\n스크리닝용 모델 저장: {MODEL_OUT} (fingerprint={best}, {NBITS}bit)")

🔎 **코드 뜯어보기 (셀 5)**
- `final.fit(Xbest, y)` : 최고 지문으로 **전체 데이터 학습**(교차검증과 달리 나누지 않고 전부 사용).
- `pickle.dump({...}, f)` : 모델과 정보를 **딕셔너리로 묶어 파일에 저장**. `open(MODEL_OUT, "wb")`=쓰기(w)+바이너리(b) 모드로 파일 열기.